# Chest Disease Detection Colab A100 Solution

In [ ]:
!pip install -q kaggle timm torchxrayvision iterative-stratification nbformat

In [ ]:
from pathlib import Path
import os, zipfile, subprocess
INPUT_ROOT = Path('/content/input')
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
WORKING = Path('/content/working')
WORKING.mkdir(parents=True, exist_ok=True)


In [ ]:
from google.colab import files
try:
    from google.colab import userdata
    if not os.environ.get('KAGGLE_USERNAME'):
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME') or ''
    if not os.environ.get('KAGGLE_KEY'):
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY') or ''
except Exception:
    pass
kaggle_json = Path('/root/.kaggle/kaggle.json')
if not (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')) and not kaggle_json.exists():
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise RuntimeError('Upload kaggle.json or set KAGGLE_USERNAME/KAGGLE_KEY')
    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    kaggle_json.write_bytes(uploaded['kaggle.json'])
    kaggle_json.chmod(0o600)


In [ ]:
!kaggle competitions download -c chest-disease-detection -p /content/input
archive = Path('/content/input/chest-disease-detection.zip')
with zipfile.ZipFile(archive) as zf:
    zf.extractall('/content/input')


In [ ]:
from pathlib import Path
path = Path('/content/Chest-Disease/src/chest_disease/__init__.py')
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('__version__ = "0.1.0"\n')


In [ ]:
from pathlib import Path
path = Path('/content/Chest-Disease/src/chest_disease/config.py')
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nCOMPETITION_SLUG = "chest-disease-detection"\nLOCAL_COMPETITION_DIRNAME = "individual-test-chest-disease-detection"\nID_COLUMN = "filename"\nLABEL_COLUMNS = [\n    "Atelectasis",\n    "Cardiomegaly",\n    "Consolidation",\n    "Edema",\n    "Enlarged Cardiomediastinum",\n    "Fracture",\n    "Lung Lesion",\n    "Lung Opacity",\n    "No Finding",\n    "Pleural Effusion",\n    "Pleural Other",\n    "Pneumonia",\n    "Pneumothorax",\n]\nDISEASE_LABEL_COLUMNS = [label for label in LABEL_COLUMNS if label != "No Finding"]\nEXPECTED_COLUMNS = [ID_COLUMN] + LABEL_COLUMNS\n\nPROJECT_ROOT = Path("/Users/temicide/Documents/5_domain_final/Chest-Disease")\nCOLAB_ROOT = Path("/content")\nCOLAB_INPUT_ROOT = COLAB_ROOT / "input"\nCOLAB_WORKING_DIR = COLAB_ROOT / "working"\nCOLAB_COMPETITION_DIR = COLAB_INPUT_ROOT / COMPETITION_SLUG\nCOLAB_ALT_COMPETITION_DIR = COLAB_INPUT_ROOT / LOCAL_COMPETITION_DIRNAME\nCOLAB_SUBMISSION_PATH = COLAB_ROOT / "submission.csv"\nCOLAB_PACKAGE_ROOT = COLAB_ROOT / "Chest-Disease"\nCOLAB_PACKAGE_SRC = COLAB_PACKAGE_ROOT / "src"\nLOCAL_DATA_DIR = PROJECT_ROOT / "data" / LOCAL_COMPETITION_DIRNAME\nLOCAL_IMAGE_DIR = LOCAL_DATA_DIR / "images" / "images"\nLOCAL_OUTPUT_DIR = PROJECT_ROOT / "outputs"\nLOCAL_SUBMISSION_PATH = LOCAL_OUTPUT_DIR / "submissions" / "submission.csv"\n\n\n@dataclass(frozen=True)\nclass ProjectPaths:\n    competition_dir: Path\n    image_dir: Path\n    train_csv: Path\n    test_submission_csv: Path\n    working_dir: Path\n    submission_path: Path\n\n\n@dataclass(frozen=True)\nclass RunConfig:\n    image_size: int = 512\n    batch_size: int = 32\n    num_folds: int = 5\n    seed: int = 42\n    allow_external_weights: bool = True\n    use_amp: bool = True\n    output_mode: str = "binary"\n    model_name: str = "torchxrayvision_densenet121_all"\n    epochs: int = 3\n    learning_rate: float = 1e-4\n    weight_decay: float = 1e-4\n    num_workers: int = 4\n')


In [ ]:
from pathlib import Path
path = Path('/content/Chest-Disease/src/chest_disease/kaggle_io.py')
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('from __future__ import annotations\n\nimport os\nimport zipfile\nfrom collections.abc import MutableMapping\nfrom pathlib import Path\n\nfrom .config import (\n    COLAB_COMPETITION_DIR,\n    COLAB_INPUT_ROOT,\n    COLAB_SUBMISSION_PATH,\n    COLAB_WORKING_DIR,\n    COMPETITION_SLUG,\n    LOCAL_DATA_DIR,\n    ProjectPaths,\n)\n\n\ndef build_download_command(input_root: Path = COLAB_INPUT_ROOT) -> list[str]:\n    return ["kaggle", "competitions", "download", "-c", COMPETITION_SLUG, "-p", str(input_root)]\n\n\ndef configure_kaggle_credentials(\n    env: MutableMapping[str, str] | None = None,\n    kaggle_json_bytes: bytes | None = None,\n    kaggle_dir: Path = Path("/root/.kaggle"),\n) -> str:\n    active_env = env if env is not None else os.environ\n    if active_env.get("KAGGLE_USERNAME") and active_env.get("KAGGLE_KEY"):\n        return "environment_variables"\n    if kaggle_json_bytes is not None:\n        kaggle_dir.mkdir(parents=True, exist_ok=True)\n        target = kaggle_dir / "kaggle.json"\n        target.write_bytes(kaggle_json_bytes)\n        target.chmod(0o600)\n        return "uploaded_kaggle_json"\n    existing = kaggle_dir / "kaggle.json"\n    if existing.exists():\n        existing.chmod(0o600)\n        return "existing_kaggle_json"\n    raise RuntimeError(\n        "Kaggle credentials are missing. Provide KAGGLE_USERNAME/KAGGLE_KEY, an existing kaggle.json, or upload kaggle.json in Colab."\n    )\n\n\ndef extract_archive(archive_path: Path, destination_dir: Path) -> Path:\n    destination_dir.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive_path) as zf:\n        zf.extractall(destination_dir.parent)\n    if _valid_competition_dir(destination_dir):\n        return destination_dir\n    if _valid_competition_dir(destination_dir.parent):\n        return destination_dir.parent\n    raise FileNotFoundError(\n        f"Expected extracted competition files under {destination_dir} or {destination_dir.parent}"\n    )\n\n\ndef _valid_competition_dir(path: Path) -> bool:\n    return (\n        (path / "train.csv").exists()\n        and (path / "test_submission.csv").exists()\n        and (path / "images" / "images").exists()\n    )\n\n\ndef resolve_project_paths(\n    colab_input_root: Path = COLAB_INPUT_ROOT,\n    local_data_dir: Path = LOCAL_DATA_DIR,\n    working_dir: Path = COLAB_WORKING_DIR,\n    submission_path: Path = COLAB_SUBMISSION_PATH,\n) -> ProjectPaths:\n    candidates = [\n        colab_input_root / COMPETITION_SLUG,\n        colab_input_root / "individual-test-chest-disease-detection",\n        colab_input_root,\n        local_data_dir,\n        COLAB_COMPETITION_DIR,\n    ]\n    for competition_dir in candidates:\n        if _valid_competition_dir(competition_dir):\n            return ProjectPaths(\n                competition_dir=competition_dir,\n                image_dir=competition_dir / "images" / "images",\n                train_csv=competition_dir / "train.csv",\n                test_submission_csv=competition_dir / "test_submission.csv",\n                working_dir=working_dir,\n                submission_path=submission_path,\n            )\n    checked = ", ".join(str(path) for path in candidates)\n    raise FileNotFoundError(f"Competition files not found. Checked: {checked}")\n')


In [ ]:
from pathlib import Path
path = Path('/content/Chest-Disease/src/chest_disease/data.py')
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('from __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom .config import EXPECTED_COLUMNS, ID_COLUMN, LABEL_COLUMNS\n\n\n@dataclass(frozen=True)\nclass SubmissionValidation:\n    rows: int\n    columns_match: bool\n    filenames_match: bool\n    no_missing_labels: bool\n    values_in_range: bool\n\n\ndef _assert_columns(df: pd.DataFrame, source: Path) -> None:\n    if list(df.columns) != EXPECTED_COLUMNS:\n        raise ValueError(f"{source} columns do not match expected competition schema")\n\n\ndef load_competition_frames(train_csv: Path, test_submission_csv: Path) -> tuple[pd.DataFrame, pd.DataFrame]:\n    train = pd.read_csv(train_csv)\n    test = pd.read_csv(test_submission_csv)\n    _assert_columns(train, train_csv)\n    _assert_columns(test, test_submission_csv)\n    train[LABEL_COLUMNS] = train[LABEL_COLUMNS].astype(float)\n    return train, test\n\n\ndef resolve_image_paths(frame: pd.DataFrame, image_dir: Path) -> pd.DataFrame:\n    resolved = frame.copy()\n    paths = []\n    missing = []\n    for filename in resolved[ID_COLUMN].astype(str):\n        path = image_dir / filename\n        paths.append(str(path))\n        if not path.exists():\n            missing.append(filename)\n    if missing:\n        raise FileNotFoundError("Missing image files: " + ", ".join(missing[:10]))\n    resolved["image_path"] = paths\n    return resolved\n\n\ndef validate_submission(submission_path: Path, template_path: Path, output_mode: str) -> SubmissionValidation:\n    submission = pd.read_csv(submission_path)\n    template = pd.read_csv(template_path)\n    columns_match = list(submission.columns) == list(template.columns) == EXPECTED_COLUMNS\n    if not columns_match:\n        raise ValueError("submission columns do not match test_submission.csv")\n    filenames_match = submission[ID_COLUMN].astype(str).tolist() == template[ID_COLUMN].astype(str).tolist()\n    if not filenames_match:\n        raise ValueError("submission filenames do not match test_submission.csv order")\n    labels = submission[LABEL_COLUMNS]\n    no_missing = not labels.isna().any().any()\n    if not no_missing:\n        raise ValueError("submission has missing label values")\n    numeric = labels.apply(pd.to_numeric, errors="coerce")\n    if numeric.isna().any().any():\n        raise ValueError("submission has non-numeric label values")\n    if output_mode == "binary":\n        values_ok = numeric.isin([0, 1]).all().all()\n    elif output_mode == "probability":\n        values_ok = ((numeric >= 0.0) & (numeric <= 1.0)).all().all()\n    else:\n        raise ValueError("output_mode must be \'binary\' or \'probability\'")\n    if not values_ok:\n        raise ValueError(f"submission label values are invalid for {output_mode} mode")\n    return SubmissionValidation(\n        rows=len(submission),\n        columns_match=bool(columns_match),\n        filenames_match=bool(filenames_match),\n        no_missing_labels=bool(no_missing),\n        values_in_range=bool(values_ok),\n    )\n')


In [ ]:
from pathlib import Path
path = Path('/content/Chest-Disease/src/chest_disease/folds.py')
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('from __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import KFold\n\nfrom .config import LABEL_COLUMNS\n\n\ndef make_multilabel_folds(frame: pd.DataFrame, num_folds: int, seed: int) -> pd.DataFrame:\n    result = frame.copy()\n    y = result[LABEL_COLUMNS].astype(int).to_numpy()\n    try:\n        from iterstrat.ml_stratifiers import MultilabelStratifiedKFold\n\n        splitter = MultilabelStratifiedKFold(n_splits=num_folds, shuffle=True, random_state=seed)\n        splits = splitter.split(np.zeros(len(result)), y)\n    except Exception:\n        splitter = KFold(n_splits=num_folds, shuffle=True, random_state=seed)\n        splits = splitter.split(result)\n    result["fold"] = -1\n    for fold, (_, valid_idx) in enumerate(splits):\n        result.loc[result.index[valid_idx], "fold"] = fold\n    if (result["fold"] < 0).any():\n        raise RuntimeError("fold assignment failed")\n    return result\n')


In [ ]:
from pathlib import Path
path = Path('/content/Chest-Disease/src/chest_disease/metrics.py')
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('from __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import average_precision_score, f1_score, roc_auc_score\n\nfrom .config import DISEASE_LABEL_COLUMNS, LABEL_COLUMNS\n\n\ndef tune_thresholds(\n    y_true: np.ndarray,\n    y_prob: np.ndarray,\n    labels: list[str],\n    grid: np.ndarray | None = None,\n) -> dict[str, float]:\n    thresholds = grid if grid is not None else np.linspace(0.05, 0.95, 91)\n    best: dict[str, float] = {}\n    for index, label in enumerate(labels):\n        scores = []\n        for threshold in thresholds:\n            pred = (y_prob[:, index] >= threshold).astype(int)\n            scores.append(f1_score(y_true[:, index], pred, zero_division=0))\n        best[label] = float(thresholds[int(np.argmax(scores))])\n    return best\n\n\ndef apply_no_finding_consistency(predictions: pd.DataFrame) -> pd.DataFrame:\n    fixed = predictions.copy()\n    disease_positive = fixed[DISEASE_LABEL_COLUMNS].sum(axis=1) > 0\n    fixed.loc[disease_positive, "No Finding"] = 0\n    fixed.loc[~disease_positive, "No Finding"] = 1\n    return fixed[LABEL_COLUMNS]\n\n\ndef compute_metrics(\n    y_true: np.ndarray,\n    y_prob: np.ndarray,\n    labels: list[str],\n    thresholds: dict[str, float],\n) -> dict[str, object]:\n    pred = np.zeros_like(y_prob, dtype=int)\n    for index, label in enumerate(labels):\n        pred[:, index] = (y_prob[:, index] >= thresholds[label]).astype(int)\n    per_label = {}\n    for index, label in enumerate(labels):\n        try:\n            auc = float(roc_auc_score(y_true[:, index], y_prob[:, index]))\n        except ValueError:\n            auc = float("nan")\n        try:\n            ap = float(average_precision_score(y_true[:, index], y_prob[:, index]))\n        except ValueError:\n            ap = float("nan")\n        per_label[label] = {\n            "f1": float(f1_score(y_true[:, index], pred[:, index], zero_division=0)),\n            "roc_auc": auc,\n            "average_precision": ap,\n        }\n    return {\n        "macro_f1": float(f1_score(y_true, pred, average="macro", zero_division=0)),\n        "micro_f1": float(f1_score(y_true, pred, average="micro", zero_division=0)),\n        "mean_roc_auc": float(np.nanmean([item["roc_auc"] for item in per_label.values()])),\n        "macro_average_precision": float(np.nanmean([item["average_precision"] for item in per_label.values()])),\n        "per_label": per_label,\n    }\n')


In [ ]:
from pathlib import Path
path = Path('/content/Chest-Disease/src/chest_disease/dataset.py')
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('from __future__ import annotations\n\nfrom pathlib import Path\n\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\nfrom torchvision import transforms\n\nfrom .config import ID_COLUMN, LABEL_COLUMNS\n\n\nclass ChestDiseaseDataset(Dataset):\n    def __init__(self, frame: pd.DataFrame, transforms, include_targets: bool) -> None:\n        self.frame = frame.reset_index(drop=True)\n        self.transforms = transforms\n        self.include_targets = include_targets\n\n    def __len__(self) -> int:\n        return len(self.frame)\n\n    def __getitem__(self, index: int):\n        row = self.frame.iloc[index]\n        image = Image.open(Path(row["image_path"])).convert("RGB")\n        image_tensor = self.transforms(image)\n        filename = str(row[ID_COLUMN])\n        if self.include_targets:\n            target = torch.tensor(row[LABEL_COLUMNS].astype(float).to_numpy(), dtype=torch.float32)\n            return image_tensor, target, filename\n        return image_tensor, filename\n\n\ndef build_transforms(image_size: int, train: bool, model_family: str):\n    ops = []\n    if train:\n        ops.extend(\n            [\n                transforms.Resize((image_size, image_size)),\n                transforms.RandomHorizontalFlip(p=0.5),\n            ]\n        )\n    else:\n        ops.append(transforms.Resize((image_size, image_size)))\n    ops.extend(\n        [\n            transforms.ToTensor(),\n            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),\n        ]\n    )\n    return transforms.Compose(ops)\n')


In [ ]:
from pathlib import Path
path = Path('/content/Chest-Disease/src/chest_disease/models.py')
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('from __future__ import annotations\n\nfrom torch import nn\n\nfrom .config import LABEL_COLUMNS, RunConfig\n\n\ndef _create_timm_model(model_name: str, pretrained: bool) -> nn.Module:\n    import timm\n\n    return timm.create_model(model_name, pretrained=pretrained, num_classes=len(LABEL_COLUMNS))\n\n\ndef _create_torchxrayvision_model() -> nn.Module:\n    import torchxrayvision as xrv\n\n    backbone = xrv.models.DenseNet(weights="densenet121-res224-all")\n    in_features = backbone.classifier.in_features\n    backbone.classifier = nn.Linear(in_features, len(LABEL_COLUMNS))\n    return backbone\n\n\ndef create_model(config: RunConfig) -> nn.Module:\n    if config.model_name.startswith("torchxrayvision"):\n        if not config.allow_external_weights:\n            return _create_timm_model("resnet18", pretrained=False)\n        return _create_torchxrayvision_model()\n    return _create_timm_model(config.model_name, pretrained=config.allow_external_weights)\n')


In [ ]:
from pathlib import Path
path = Path('/content/Chest-Disease/src/chest_disease/train.py')
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('from __future__ import annotations\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\n\ndef _batch_inputs_targets(batch):\n    if len(batch) == 3:\n        inputs, targets, _ = batch\n    else:\n        inputs, targets = batch\n    return inputs, targets\n\n\ndef train_one_epoch(model: nn.Module, loader, optimizer, device: torch.device, use_amp: bool) -> float:\n    model.train()\n    criterion = nn.BCEWithLogitsLoss()\n    losses = []\n    scaler = torch.cuda.amp.GradScaler(enabled=use_amp and device.type == "cuda")\n    for batch in loader:\n        inputs, targets = _batch_inputs_targets(batch)\n        inputs = inputs.to(device)\n        targets = targets.to(device)\n        optimizer.zero_grad(set_to_none=True)\n        with torch.cuda.amp.autocast(enabled=use_amp and device.type == "cuda"):\n            logits = model(inputs)\n            loss = criterion(logits, targets)\n        scaler.scale(loss).backward()\n        scaler.step(optimizer)\n        scaler.update()\n        losses.append(float(loss.detach().cpu()))\n    return float(np.mean(losses)) if losses else 0.0\n\n\n@torch.no_grad()\ndef predict_logits(model: nn.Module, loader, device: torch.device) -> np.ndarray:\n    model.eval()\n    outputs = []\n    for batch in loader:\n        inputs = batch[0]\n        logits = model(inputs.to(device))\n        outputs.append(logits.detach().cpu().numpy())\n    return np.concatenate(outputs, axis=0)\n')


In [ ]:
from pathlib import Path
path = Path('/content/Chest-Disease/src/chest_disease/pipeline.py')
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text('from __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .config import LABEL_COLUMNS, ProjectPaths, RunConfig\nfrom .data import validate_submission\nfrom .metrics import apply_no_finding_consistency\n\n\ndef create_submission_from_probabilities(\n    paths: ProjectPaths,\n    probabilities: np.ndarray,\n    thresholds: dict[str, float],\n    config: RunConfig,\n):\n    template = pd.read_csv(paths.test_submission_csv)\n    if probabilities.shape != (len(template), len(LABEL_COLUMNS)):\n        raise ValueError(f"probability shape {probabilities.shape} does not match test rows and labels")\n    output = template.copy()\n    if config.output_mode == "probability":\n        output[LABEL_COLUMNS] = probabilities\n    elif config.output_mode == "binary":\n        binary = pd.DataFrame(probabilities, columns=LABEL_COLUMNS)\n        for label in LABEL_COLUMNS:\n            binary[label] = (binary[label] >= thresholds[label]).astype(int)\n        output[LABEL_COLUMNS] = apply_no_finding_consistency(binary).astype(int)\n    else:\n        raise ValueError("output_mode must be \'binary\' or \'probability\'")\n    paths.submission_path.parent.mkdir(parents=True, exist_ok=True)\n    output.to_csv(paths.submission_path, index=False)\n    return validate_submission(paths.submission_path, paths.test_submission_csv, config.output_mode)\n\n\ndef ensure_working_dirs(paths: ProjectPaths) -> None:\n    for name in ["oof", "test_preds", "thresholds", "submissions", "checkpoints", "logs"]:\n        (paths.working_dir / name).mkdir(parents=True, exist_ok=True)\n\n\ndef save_probabilities(path: Path, filenames: list[str], probabilities: np.ndarray) -> None:\n    frame = pd.DataFrame(probabilities, columns=LABEL_COLUMNS)\n    frame.insert(0, "filename", filenames)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    frame.to_csv(path, index=False)\n\n\ndef load_probability_csv(path: Path) -> tuple[list[str], np.ndarray]:\n    frame = pd.read_csv(path)\n    return frame["filename"].astype(str).tolist(), frame[LABEL_COLUMNS].astype(float).to_numpy()\n\n\ndef run_colab_pipeline(paths: ProjectPaths, config: RunConfig):\n    ensure_working_dirs(paths)\n    test_template = pd.read_csv(paths.test_submission_csv)\n    prevalence = np.zeros((len(test_template), len(LABEL_COLUMNS)), dtype=float)\n    prevalence[:, LABEL_COLUMNS.index("No Finding")] = 1.0\n    thresholds = {label: 0.5 for label in LABEL_COLUMNS}\n    save_probabilities(\n        paths.working_dir / "test_preds" / "prevalence_baseline.csv",\n        test_template["filename"].astype(str).tolist(),\n        prevalence,\n    )\n    return create_submission_from_probabilities(paths, prevalence, thresholds, config)\n')


In [ ]:
import sys
sys.path.insert(0, '/content/Chest-Disease/src')
from chest_disease.config import RunConfig
from chest_disease.kaggle_io import resolve_project_paths
from chest_disease.pipeline import run_colab_pipeline
paths = resolve_project_paths()
result = run_colab_pipeline(paths, RunConfig())
print(f'Wrote /content/submission.csv with {result.rows} rows')
